In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# INAIL - SISTEMA DI ARRICCHIMENTO OTTIMALE
# TF-IDF + LLM Post-Processing (Best of Both Worlds)

# SETUP
%pip install -q scikit-learn ollama

import json
import re
import time
import logging
import subprocess
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple
from datetime import datetime
from collections import Counter

# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
# CONFIGURAZIONE

class EnrichmentConfig:
    INAIL_DATA_DIR = Path('/content/drive/MyDrive/INAIL_Thesis_Data')
    JSON_DIR = INAIL_DATA_DIR / 'json'
    ENRICHED_DIR = INAIL_DATA_DIR / 'enriched_json'
    LOG_DIR = INAIL_DATA_DIR / 'logs'

    OLLAMA_MODELS = ["mistral:latest", "llama3.1:latest", "llama3:latest"]

    DOCUMENT_CATEGORIES = [
        "Prevenzione incendi", "Sicurezza sul lavoro",
        "Infortuni e malattie professionali", "Normativa e regolamenti tecnici",
        "Formazione e istruzione", "Valutazione rischi",
        "DPI e dispositivi di protezione", "Edilizia e cantieri",
        "Agricoltura", "Industria manifatturiera",
        "Sanità e strutture sanitarie", "Altro"
    ]

    MIN_TEXT_LENGTH = 100

    @classmethod
    def setup(cls):
        for d in [cls.ENRICHED_DIR, cls.LOG_DIR]:
            d.mkdir(exist_ok=True, parents=True)

In [4]:
# ESTRATTORE LEGGI
# =============================================================================

class ItalianLawExtractor:

    # Estrattore completo di riferimenti normativi italiani
    PATTERNS = {
        'dlgs': [
            r'[Dd]\.?\s*[Ll]\.?\s*[Gg]\.?\s*[Ss]?\.?\s*(?:n\.?\s*)?(\d+)\s*[/\-°]\s*(\d{2,4})',
        ],
        'dm': [
            r'[Dd]\.?\s*[Mm]\.?\s+(?:del\s+)?(\d{1,2})[/\-\s]+(\d{1,2})[/\-\s]+(\d{2,4})',
        ],
        'legge': [
            r'[Ll]\.?\s*(?:n\.?\s*)?(\d+)\s*[/\-°]\s*(\d{2,4})',
        ],
        'dpr': [
            r'[Dd]\.?\s*[Pp]\.?\s*[Rr]\.?\s*(?:n\.?\s*)?(\d+)\s*[/\-°]\s*(\d{2,4})',
        ],
        'direttiva_ue': [
            r'[Dd]irettiva\s+(?:UE\s+)?(\d{4})\s*[/\-]\s*(\d+)',
        ],
    }

    def __init__(self):
        self.compiled_patterns = {}
        for tipo, patterns in self.PATTERNS.items():
            self.compiled_patterns[tipo] = [re.compile(p, re.IGNORECASE) for p in patterns]

    def extract(self, text: str) -> List[str]:
        found_laws = set()
        for tipo, compiled_patterns in self.compiled_patterns.items():
            for pattern in compiled_patterns:
                matches = pattern.findall(text)
                for match in matches:
                    normalized = self._normalize_law(tipo, match)
                    if normalized:
                        found_laws.add(normalized)
        return sorted(found_laws, key=lambda x: (len(x), 'D.Lgs' in x), reverse=True)

    def _normalize_law(self, tipo: str, match: tuple) -> Optional[str]:
        try:
            if tipo == 'dlgs':
                num, anno = match[0], match[1]
                anno = self._normalize_year(anno)
                return f"D.Lgs. {num}/{anno}"
            elif tipo == 'dm':
                if len(match) == 3 and match[1].isdigit():
                    giorno, mese, anno = match
                    anno = self._normalize_year(anno)
                    return f"D.M. {giorno}/{mese}/{anno}"
            elif tipo == 'legge':
                num, anno = match
                anno = self._normalize_year(anno)
                return f"Legge n. {num}/{anno}"
            elif tipo == 'dpr':
                num, anno = match[0], match[1]
                anno = self._normalize_year(anno)
                return f"D.P.R. {num}/{anno}"
            elif tipo == 'direttiva_ue':
                anno, num = match
                return f"Direttiva UE {anno}/{num}"
        except:
            return None
        return None

    def _normalize_year(self, year_str: str) -> str:
        year = int(year_str)
        if year < 100:
            year = 2000 + year if year < 50 else 1900 + year
        return str(year)

In [5]:
# TF-IDF KEYWORD EXTRACTOR

class TFIDFKeywordExtractor:

    # Estrattore TF-IDF ottimizzato per italiano tecnico
    def __init__(self):
        print("[TF-IDF] Inizializzazione...")

        # Stopwords italiane estese
        self.italian_stopwords = [
            'il', 'lo', 'la', 'i', 'gli', 'le', 'un', 'uno', 'una',
            'di', 'a', 'da', 'in', 'con', 'su', 'per', 'tra', 'fra',
            'degli', 'delle', 'della', 'dello', 'nei', 'sui', 'ai',
            'come', 'più', 'anche', 'essere', 'avere', 'fare', 'dire',
            'questo', 'quello', 'stesso', 'altro', 'molto', 'tutto',
            'ogni', 'tale', 'quale', 'quando', 'dove', 'che', 'chi',
            'cui', 'cosa', 'quanto', 'già', 'ancora', 'prima', 'dopo',
            'sempre', 'mai', 'oggi', 'ieri', 'domani', 'ora', 'poi',
            'così', 'quindi', 'mentre', 'però', 'invece', 'inoltre'
        ]

        # Blacklist termini troppo generici INAIL
        self.blacklist = {
            'identificazione', 'misure', 'gestione', 'controllo',
            'procedure', 'strumenti', 'attività', 'modalità',
            'presente', 'documento', 'capitolo', 'sezione',
            'figura', 'tabella', 'allegato', 'riferimento',
            'sicurezza', 'lavoro', 'rischio', 'prevenzione',  # Troppo generici
            'protezione', 'normativa', 'dispositivo', 'utilizzo',
            'soggetto', 'oggetto', 'elemento', 'componente'
        }

        print("[TF-IDF] ✓ Pronto")

    def extract_keywords(
        self,
        text: str,
        top_n: int = 20
    ) -> List[Tuple[str, float]]:

        # Estrae keyword con TF-IDF
        if len(text) < EnrichmentConfig.MIN_TEXT_LENGTH:
            return []

        try:
            # Pulisci testo
            text_cleaned = re.sub(r'\s+', ' ', text).strip()

            # Limita lunghezza per performance (primi 50k char)
            if len(text_cleaned) > 50000:
                # Prendi: inizio + metà + fine
                chunks = [
                    text_cleaned[:15000],
                    text_cleaned[len(text_cleaned)//2 - 7500:len(text_cleaned)//2 + 7500],
                    text_cleaned[-15000:]
                ]
                text_cleaned = ' '.join(chunks)

            # TF-IDF con parametri ottimizzati
            vectorizer = TfidfVectorizer(
                max_features=100,
                ngram_range=(1, 3),  # unigram, bigram, trigram
                stop_words=self.italian_stopwords,
                min_df=1,
                max_df=1.0,
                lowercase=True,
                token_pattern=r'(?u)\b[a-zàèéìòù]+\b'  # Solo lettere italiane
            )

            # Fit-transform
            tfidf_matrix = vectorizer.fit_transform([text_cleaned])
            feature_names = vectorizer.get_feature_names_out()
            scores = tfidf_matrix.toarray()[0]

            # Ordina per score
            keyword_scores = [
                (feature_names[i], scores[i])
                for i in range(len(feature_names))
                if scores[i] > 0
            ]
            keyword_scores.sort(key=lambda x: x[1], reverse=True)

            # Filtra con blacklist
            filtered = self._filter_keywords(keyword_scores)

            return filtered[:top_n]

        except Exception as e:
            print(f"[TF-IDF] Errore: {e}")
            return []

    def _filter_keywords(
        self,
        keywords: List[Tuple[str, float]]
    ) -> List[Tuple[str, float]]:

        # Filtra keyword generiche o invalide
        filtered = []
        for kw, score in keywords:
            # Skip se troppo corto
            if len(kw) < 4:
                continue

            # Skip se in blacklist
            if kw.lower() in self.blacklist:
                continue

            # Skip se inizia con numero
            if kw[0].isdigit():
                continue

            # Skip se solo numeri
            if kw.replace(' ', '').isdigit():
                continue

            filtered.append((kw, score))

        return filtered

In [6]:
# LLM POST-PROCESSOR

class LLMKeywordRefiner:

    # Raffina keyword TF-IDF con una singola chiamata LLM
    def __init__(self, model: str = "mistral:latest", base_url: str = "http://localhost:11434"):
        self.model = model
        self.base_url = base_url

    def refine_keywords(
        self,
        tfidf_keywords: List[str],
        document_title: str,
        logger: logging.Logger = None
    ) -> List[str]:

        # Raffina keyword TF-IDF in 2 categorie:
          # Tecniche (mantieni)
          # Generali (estrai dal contesto)


        if not tfidf_keywords:
            return []

        # Prendi top 15 da TF-IDF
        top_tfidf = tfidf_keywords[:15]

        prompt = f"""Sei un esperto INAIL. Analizza queste keyword estratte dal documento "{document_title}".

KEYWORD TF-IDF:
{', '.join(top_tfidf)}

COMPITO: Seleziona le 10 migliori keyword bilanciando:
- 6 keyword TECNICHE specifiche (sostanze, procedure, dispositivi, metodi)
- 4 keyword GENERALI di contesto (temi, ambiti applicativi)

REGOLE:
- Mantieni keyword tecniche esatte da TF-IDF (es: "cromo esavalente", "FFP3")
- Aggiungi 2-3 keyword generali dedotte dal contesto
- EVITA: "sicurezza", "rischio", "prevenzione" (troppo generici)
- Massimo 10 keyword totali

Rispondi SOLO con lista puntata, una keyword per riga."""

        try:
            import requests
            response = requests.post(
                f"{self.base_url}/api/generate",
                json={
                    "model": self.model,
                    "prompt": prompt,
                    "stream": False,
                    "temperature": 0.2
                },
                timeout=60
            )

            if response.status_code != 200:
                if logger:
                    logger.warning("  [LLM Refiner] Fallback a TF-IDF raw")
                return top_tfidf[:10]

            output = response.json().get("response", "")
            refined = self._parse_keyword_list(output)

            # Se parsing fallisce, usa TF-IDF
            if not refined:
                return top_tfidf[:10]

            return refined[:10]

        except Exception as e:
            if logger:
                logger.warning(f"  [LLM Refiner] Error: {e}, usando TF-IDF")
            return top_tfidf[:10]

    def _parse_keyword_list(self, text: str) -> List[str]:

        # Parse lista keyword dall'output LLM con pulizia avanzata
        lines = text.strip().split("\n")
        keywords = []

        for line in lines:
            line = line.strip()

            # Rimuovi numerazione e bullet points (più aggressivo)
            line = re.sub(r"^\d+[\.\)]\s*", "", line)  # 1. o 1)
            line = re.sub(r"^[\-\*\•]\s*", "", line)   # - o * o •
            line = re.sub(r"^\.+\s*", "", line)        # . iniziale

            # Skip se vuoto o troppo corto/lungo
            if not line or len(line) < 3 or len(line) > 60:
                continue

            # Skip se contiene troppi caratteri speciali
            if line.count('.') > 2 or line.count('(') > 1:
                continue

            # Normalizza
            line = line.lower().strip()

            # Skip se inizia con numero (residuo parsing)
            if line[0].isdigit():
                continue

            keywords.append(line)

        # Dedup mantenendo ordine
        return list(dict.fromkeys(keywords))

In [7]:
# OLLAMA CLIENT

class OllamaClient:
    def __init__(self, model: str = "mistral:latest"):
        self.model = model
        self.provider = "ollama"
        self.base_url = "http://localhost:11434"

    def generate(self, prompt: str, max_tokens: int = 2000) -> Optional[str]:
        try:
            import requests
            response = requests.post(
                f"{self.base_url}/api/generate",
                json={
                    "model": self.model,
                    "prompt": prompt,
                    "stream": False,
                    "temperature": 0.1
                },
                timeout=120
            )
            return response.json()['response'] if response.status_code == 200 else None
        except Exception as e:
            print(f"Ollama error: {e}")
            return None

In [8]:
# UTILITY FUNCTIONS

def extract_full_document_text(document_data: Dict[str, Any]) -> str:
    # Estrae tutto il testo del documento
    chunks = []

    abstract = document_data.get('web_metadata', {}).get('abstract', '')
    if abstract:
        chunks.append(f"ABSTRACT:\n{abstract}\n")

    content = document_data.get('document_content', {})
    plain_text = content.get('plain_text', '')
    if plain_text:
        chunks.append(plain_text)
    elif content.get('markdown_content'):
        chunks.append(content['markdown_content'])

    return '\n\n'.join(chunks)

def setup_logging(log_dir: Path):

    # Setup logging
    log_dir.mkdir(exist_ok=True, parents=True)
    log_file = log_dir / f"enrichment_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[logging.FileHandler(log_file), logging.StreamHandler()]
    )

    return logging.getLogger(__name__), log_file

In [9]:
# PROMPT BUILDING

def build_enhanced_prompt(
    document_data: Dict[str, Any],
    extracted_laws: List[str],
    refined_keywords: List[str]
) -> str:

    # Prompt con hint da regex e keyword raffinate
    title = document_data.get('web_metadata', {}).get('title', 'N/A')
    full_text = extract_full_document_text(document_data)
    content_text = full_text[:6000]

    laws_hint = ""
    if extracted_laws:
        laws_hint = f"\n\n[LEGGI IDENTIFICATE: {', '.join(extracted_laws[:10])}]"

    keywords_hint = ""
    if refined_keywords:
        keywords_hint = f"\n[KEYWORD PRINCIPALI: {', '.join(refined_keywords[:8])}]"

    prompt = f"""Sei un esperto di normative INAIL. Analizza questo documento tecnico.

DOCUMENTO: {title}
TESTO: {content_text}{laws_hint}{keywords_hint}

Rispondi con ESATTAMENTE 5 righe numerate:

1. SINTESI: Una frase di 25-40 parole che riassume il contenuto principale

2. CATEGORIA: Scegli UNA tra {', '.join(EnrichmentConfig.DOCUMENT_CATEGORIES[:6])}

3. LEGGI: Conferma o integra le leggi identificate. Se nessuna scrivi "NESSUNA"

4. PROFESSIONI: Figure professionali destinatarie. Se non chiaro scrivi "NESSUNA"

5. TEMI: 3-5 temi principali separati da virgola

Sii preciso e tecnico."""

    return prompt

In [10]:
# RESPONSE PARSING

def parse_enhanced_llm_response(
    response: str,
    regex_laws: List[str],
    refined_keywords: List[str]
) -> Dict[str, Any]:

    # Parse risposta LLM
    data = {
        'sintesi': 'N/A',
        'categoria_principale': 'Altro',
        'categorie_secondarie': [],
        'articoli_legge': [],
        'categorie_professionali': [],
        'parole_chiave': refined_keywords,  # Keyword già raffinate
        'temi_principali': [],
        'extraction_sources': {
            'laws': 'none',
            'keywords': 'tfidf_llm_refined'
        }
    }

    lines = response.strip().split('\n')

    for line in lines:
        line = line.strip()

        if re.match(r'^1\.', line):
            sintesi = re.split(r'SINTESI:?', line, flags=re.IGNORECASE)[-1].strip()
            if sintesi and len(sintesi) > 10:
                data['sintesi'] = sintesi[:500]

        elif re.match(r'^2\.', line):
            cat = re.split(r'CATEGORIA:?', line, flags=re.IGNORECASE)[-1].strip()
            if cat and cat not in ['nessuna', 'altro', 'na']:
                data['categoria_principale'] = cat

        elif re.match(r'^3\.', line):
            leggi_str = re.split(r'LEGGI:?', line, flags=re.IGNORECASE)[-1].strip()
            if leggi_str.lower() not in ['nessuna', 'na']:
                llm_laws = [x.strip() for x in re.split(r'[,;]', leggi_str) if x.strip()]
                data['articoli_legge'] = llm_laws
                data['extraction_sources']['laws'] = 'llm'

        elif re.match(r'^4\.', line):
            prof_str = re.split(r'PROFESSIONI?:', line, flags=re.IGNORECASE)[-1].strip()
            if prof_str.lower() not in ['nessuna', 'na', 'nessuno']:
                profs = [x.strip() for x in re.split(r'[,;]', prof_str) if x.strip() and len(x.strip()) > 3]
                data['categorie_professionali'] = profs[:5]

        elif re.match(r'^5\.', line):
            temi_str = re.split(r'TEMI:?', line, flags=re.IGNORECASE)[-1].strip()
            data['temi_principali'] = [x.strip() for x in temi_str.split(',') if x.strip()][:5]

    # Regex laws hanno priorità
    if regex_laws:
        data['articoli_legge'] = regex_laws
        data['extraction_sources']['laws'] = 'regex_primary'

    return data

In [11]:
# CONFIDENCE CALCULATOR

class ConfidenceCalculator:
    @staticmethod
    def calculate_comprehensive_confidence(
        document_data: Dict[str, Any],
        extracted_laws: List[str],
        metadata: Dict[str, Any],
        logger: logging.Logger
    ) -> Dict[str, Any]:

        scores = {}
        review_reasons = []

        # 1) QUALITÀ TESTO
        full_text = extract_full_document_text(document_data)
        text_length = len(full_text)

        if text_length < 500:
            scores['text_quality'] = 0.3
        elif text_length < 2000:
            scores['text_quality'] = 0.7
        else:
            scores['text_quality'] = 0.95

        # 2) ESTRAZIONE LEGGI
        if len(extracted_laws) >= 3:
            scores['law_extraction'] = 0.95
        elif len(extracted_laws) >= 1:
            scores['law_extraction'] = 0.75
        else:
            scores['law_extraction'] = 0.4
            review_reasons.append("Poche leggi estratte")

        # 3) KEYWORD QUALITY (più severo con validazione)
        keywords = metadata.get('parole_chiave', [])
        kw_count = len(keywords)

        # Controlla qualità keyword (no spazzatura)
        valid_keywords = [
            kw for kw in keywords
            if len(kw) >= 4
            and not re.match(r'^[\d\.\s]+', kw)  # No numeri/punti iniziali
            and kw.count('.') <= 1  # Max 1 punto
        ]
        valid_count = len(valid_keywords)

        if valid_count >= 8:
            scores['keyword_quality'] = 0.95
        elif valid_count >= 6:
            scores['keyword_quality'] = 0.80
        elif valid_count >= 4:
            scores['keyword_quality'] = 0.65
        else:
            scores['keyword_quality'] = 0.4
            review_reasons.append(f"Solo {valid_count} keyword valide su {kw_count}")

        # 4) COMPLETEZZA METADATI
        required_fields = ['sintesi', 'categoria_principale', 'parole_chiave', 'temi_principali']
        filled = sum(1 for f in required_fields if metadata.get(f) and metadata[f] not in ['N/A', [], 'Altro'])
        scores['metadata_completeness'] = filled / len(required_fields)

        if filled < 3:
            review_reasons.append("Metadati incompleti")

        # 5) VALIDITÀ CATEGORIA
        if metadata.get('categoria_principale', 'Altro') in EnrichmentConfig.DOCUMENT_CATEGORIES:
            scores['category_validity'] = 1.0
        else:
            scores['category_validity'] = 0.5
            review_reasons.append("Categoria non standard")

        # 6) QUALITÀ SINTESI (fix: limite più generoso)
        sintesi = metadata.get('sintesi', '')
        sintesi_words = len(sintesi.split())

        # Valuta in base a PAROLE, non caratteri
        if 20 <= sintesi_words <= 60:  # 20-60 parole = range ottimale
            scores['sintesi_quality'] = 1.0
        elif 10 <= sintesi_words <= 80:  # Range accettabile
            scores['sintesi_quality'] = 0.9
        elif sintesi_words >= 80:  # Troppo lunga ma completa
            scores['sintesi_quality'] = 0.75
        elif sintesi_words >= 5:  # Troppo corta ma presente
            scores['sintesi_quality'] = 0.5
        else:
            scores['sintesi_quality'] = 0.2
            review_reasons.append("Sintesi mancante o troppo breve")

        # OVERALL
        overall = sum(scores.values()) / len(scores)

        # LIVELLO
        if overall >= 0.85:
            level = "HIGH"
            needs_review = False
        elif overall >= 0.70:
            level = "MEDIUM"
            needs_review = len(review_reasons) > 2
        else:
            level = "LOW"
            needs_review = True

        return {
            'overall_score': round(overall, 3),
            'confidence_level': level,
            'component_scores': {k: round(v, 3) for k, v in scores.items()},
            'needs_review': needs_review,
            'review_reasons': review_reasons
        }

In [12]:
# DOCUMENT ENRICHMENT

def enrich_document_enhanced(
    document_data: Dict[str, Any],
    llm_client: OllamaClient,
    law_extractor: ItalianLawExtractor,
    keyword_extractor: TFIDFKeywordExtractor,
    keyword_refiner: LLMKeywordRefiner,
    confidence_calculator: ConfidenceCalculator,
    logger: logging.Logger
) -> Optional[Dict[str, Any]]:

    # Pipeline completa di enrichment
    doc_title = document_data.get('web_metadata', {}).get('title', 'N/A')
    logger.info(f"\nProcessing: {doc_title[:60]}...")

    try:
        # 1) ESTRAI TESTO
        full_text = extract_full_document_text(document_data)

        logger.info(f"  [Testo] Lunghezza: {len(full_text)} caratteri")

        if len(full_text) < EnrichmentConfig.MIN_TEXT_LENGTH:
            logger.warning(f"  Testo troppo breve, skip")
            return None

        # 2) ESTRAI LEGGI CON REGEX
        extracted_laws = law_extractor.extract(full_text)
        logger.info(f"  [Regex] {len(extracted_laws)} leggi trovate")

        # 3) ESTRAI KEYWORD CON TF-IDF (veloce)
        tfidf_keywords = keyword_extractor.extract_keywords(full_text, top_n=20)
        tfidf_kw_list = [kw for kw, _ in tfidf_keywords]
        logger.info(f"  [TF-IDF] {len(tfidf_keywords)} keyword estratte")
        for kw, score in tfidf_keywords[:5]:
            logger.info(f"    • {kw} ({score:.3f})")

        # 4) RAFFINA KEYWORD CON LLM (1 chiamata singola)
        refined_keywords = keyword_refiner.refine_keywords(
            tfidf_kw_list,
            doc_title,
            logger
        )

        # VALIDAZIONE: Se LLM refiner ha fallito, usa TF-IDF raw
        if not refined_keywords or len(refined_keywords) < 3:
            logger.warning("  [LLM Refiner] Output invalido, uso TF-IDF raw")
            refined_keywords = tfidf_kw_list[:10]

        # Filtra keyword con caratteri strani residui
        refined_keywords = [
            kw for kw in refined_keywords
            if not re.match(r'^[\d\.\s]+', kw)  # Skip se inizia con numeri/punti
            and len(kw) >= 4
            and kw.count('.') <= 1  # Max 1 punto (es: "D.Lgs")
        ]

        # Se troppo poche dopo filtering, integra con TF-IDF
        if len(refined_keywords) < 5:
            logger.warning(f"  [LLM Refiner] Solo {len(refined_keywords)} keyword valide, integro con TF-IDF")
            # Aggiungi keyword TF-IDF che non sono già presenti
            for kw, _ in tfidf_keywords[:15]:
                if kw not in refined_keywords and len(refined_keywords) < 10:
                    refined_keywords.append(kw)

        logger.info(f"  [LLM Refiner] {len(refined_keywords)} keyword finali")
        for kw in refined_keywords[:5]:
            logger.info(f"    • {kw}")

        # 5) GENERA PROMPT PRINCIPALE
        prompt = build_enhanced_prompt(document_data, extracted_laws, refined_keywords)

        # 6) CHIAMA LLM PER METADATI
        response = llm_client.generate(prompt, max_tokens=1500)
        if not response:
            logger.error("  LLM non ha risposto")
            return None

        # 7) PARSE RISPOSTA
        metadata = parse_enhanced_llm_response(response, extracted_laws, refined_keywords)

        # 8) CALCOLA CONFIDENCE
        confidence_data = confidence_calculator.calculate_comprehensive_confidence(
            document_data,
            extracted_laws,
            metadata,
            logger
        )

        # 9) LOG RISULTATI
        logger.info(f"  [Risultati]")
        logger.info(f"    Sintesi: {metadata['sintesi'][:80]}...")
        logger.info(f"    Categoria: {metadata['categoria_principale']}")
        logger.info(f"    Leggi ({len(metadata['articoli_legge'])}): {metadata['articoli_legge'][:3]}")
        logger.info(f"    Keywords ({len(metadata['parole_chiave'])}): {metadata['parole_chiave'][:5]}")
        logger.info(f"    Confidence: {confidence_data['overall_score']:.2f} ({confidence_data['confidence_level']})")
        logger.info(f"    Needs Review: {'YES' if confidence_data['needs_review'] else 'NO'}")

        # 10) CREA DOCUMENTO ARRICCHITO
        enriched_doc = document_data.copy()
        enriched_doc['semantic_metadata'] = {
            **metadata,
            'confidence': confidence_data,
            'generated_at': datetime.now().isoformat(),
            'llm_model': llm_client.model,
            'llm_provider': llm_client.provider,
            'keyword_method': 'tfidf_llm_refined',
            'version': 'hybrid_v1.0'
        }

        return enriched_doc

    except Exception as e:
        logger.error(f"  Errore durante enrichment: {e}")
        import traceback
        traceback.print_exc()
        return None

In [13]:
# BATCH ENRICHMENT

def batch_enrich_documents(
    llm_client: OllamaClient,
    law_extractor: ItalianLawExtractor,
    keyword_extractor: TFIDFKeywordExtractor,
    keyword_refiner: LLMKeywordRefiner,
    confidence_calculator: ConfidenceCalculator,
    max_documents: Optional[int] = None,
    logger: logging.Logger = None
) -> Dict[str, Any]:
    # Batch enrichment con statistiche

    json_dir = EnrichmentConfig.JSON_DIR
    enriched_dir = EnrichmentConfig.ENRICHED_DIR

    json_files = sorted([
        f for f in json_dir.glob("*.json")
        if not f.name.startswith('vector_db')
    ])

    if max_documents:
        json_files = json_files[:max_documents]

    logger.info(f"\n{'='*80}")
    logger.info(f"BATCH ENRICHMENT - TF-IDF + LLM HYBRID")
    logger.info(f"{'='*80}")
    logger.info(f"Documenti: {len(json_files)}")
    logger.info(f"LLM: {llm_client.provider}/{llm_client.model}")
    logger.info(f"Keyword: TF-IDF + LLM Refiner")
    logger.info(f"{'='*80}\n")

    stats = {
        'total': len(json_files),
        'enriched': 0,
        'failed': 0,
        'skipped': 0,
        'total_laws': 0,
        'total_keywords': 0,
        'confidence_distribution': {'HIGH': 0, 'MEDIUM': 0, 'LOW': 0},
        'needs_review': 0,
        'processing_time': 0
    }

    start_time = time.time()

    for idx, json_file in enumerate(json_files, 1):
        logger.info(f"\n[{idx}/{len(json_files)}] {json_file.name[:60]}")

        enriched_path = enriched_dir / json_file.name

        if enriched_path.exists():
            logger.info("  Già processato, skip")
            stats['skipped'] += 1
            continue

        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                doc_data = json.load(f)

            enriched_doc = enrich_document_enhanced(
                doc_data,
                llm_client,
                law_extractor,
                keyword_extractor,
                keyword_refiner,
                confidence_calculator,
                logger
            )

            if enriched_doc:
                with open(enriched_path, 'w', encoding='utf-8') as f:
                    json.dump(enriched_doc, f, ensure_ascii=False, indent=2)

                metadata = enriched_doc['semantic_metadata']
                confidence = metadata['confidence']

                stats['enriched'] += 1
                stats['total_laws'] += len(metadata['articoli_legge'])
                stats['total_keywords'] += len(metadata['parole_chiave'])
                stats['confidence_distribution'][confidence['confidence_level']] += 1

                if confidence['needs_review']:
                    stats['needs_review'] += 1

                logger.info(f"  Salvato ({confidence['confidence_level']})")
            else:
                stats['failed'] += 1
                logger.warning(f"  Fallito")

            time.sleep(1)

        except KeyboardInterrupt:
            logger.warning(f"\n INTERRUZIONE MANUALE")
            break

        except Exception as e:
            logger.error(f"  Errore: {e}")
            stats['failed'] += 1

    stats['processing_time'] = time.time() - start_time

    # REPORT FINALE
    logger.info(f"\n{'='*80}")
    logger.info(f"REPORT FINALE")
    logger.info(f"{'='*80}")
    logger.info(f"Totale: {stats['total']}")
    logger.info(f" Arricchiti: {stats['enriched']}")
    logger.info(f" Già processati: {stats['skipped']}")
    logger.info(f"Falliti: {stats['failed']}")
    logger.info(f"\nSTATISTICHE:")
    logger.info(f"  Leggi totali: {stats['total_laws']}")
    logger.info(f"  Keywords totali: {stats['total_keywords']}")
    if stats['enriched'] > 0:
        logger.info(f"  Media leggi/doc: {stats['total_laws']/stats['enriched']:.1f}")
        logger.info(f"  Media keywords/doc: {stats['total_keywords']/stats['enriched']:.1f}")
    logger.info(f"\nCONFIDENCE:")
    for level in ['HIGH', 'MEDIUM', 'LOW']:
        count = stats['confidence_distribution'][level]
        pct = (count / stats['enriched'] * 100) if stats['enriched'] > 0 else 0
        logger.info(f"  {level}: {count} ({pct:.1f}%)")
    logger.info(f"\n Da rivedere: {stats['needs_review']}")
    logger.info(f" Tempo: {stats['processing_time']/60:.1f} minuti")
    logger.info(f"{'='*80}\n")

    return stats

In [14]:
# OLLAMA SETUP

def install_ollama():
    print("[Setup] Installazione Ollama...")
    try:
        subprocess.run([
            'bash', '-c',
            'curl -fsSL https://ollama.ai/install.sh | sh'
        ], check=True, capture_output=True, timeout=300)
        print("Ollama installato")
        return True
    except Exception as e:
        print(f"Errore: {e}")
        return False

def start_ollama():
    print("[Setup] Avvio server Ollama...")
    try:
        subprocess.Popen(
            ['ollama', 'serve'],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        time.sleep(5)
        print("Server avviato")
        return True
    except Exception as e:
        print(f"Errore: {e}")
        return False

def pull_best_model() -> Optional[str]:
    print("[Setup] Download modello LLM...")
    for model in EnrichmentConfig.OLLAMA_MODELS:
        try:
            print(f"  Tentando {model}...")
            result = subprocess.run(
                ['ollama', 'pull', model],
                capture_output=True,
                timeout=300,
                text=True
            )
            if result.returncode == 0:
                print(f"  {model} pronto!")
                return model
        except:
            continue
    return None

In [15]:
# TEST RAPIDO

def quick_test():
    # Test su singolo documento

    print("\n" + "="*80)
    print("TEST SISTEMA ENRICHMENT - TF-IDF + LLM HYBRID")
    print("="*80 + "\n")

    EnrichmentConfig.setup()
    logger, log_file = setup_logging(EnrichmentConfig.LOG_DIR)

    # Setup LLM
    print("Configurazione LLM...")
    if not install_ollama() or not start_ollama():
        print("Ollama setup fallito")
        return

    model = pull_best_model()
    if not model:
        print("Nessun modello disponibile")
        return

    llm_client = OllamaClient(model=model)

    # Setup componenti
    print("\n[Setup] Inizializzazione componenti...")
    law_extractor = ItalianLawExtractor()
    keyword_extractor = TFIDFKeywordExtractor()
    keyword_refiner = LLMKeywordRefiner(model=model)
    confidence_calculator = ConfidenceCalculator()
    print("Tutti i componenti pronti\n")

    # Trova documenti
    json_dir = EnrichmentConfig.JSON_DIR
    json_files = sorted([
        f for f in json_dir.glob("*.json")
        if not f.name.startswith('vector_db')
    ])

    if not json_files:
        print("Nessun documento JSON trovato")
        return

    print(f"Documenti disponibili: {len(json_files)}\n")
    for i, f in enumerate(json_files[:10], 1):
        print(f"{i}. {f.name[:70]}")

    idx = int(input(f"\nQuale testare? (1-{min(10, len(json_files))}): ").strip() or "1") - 1
    test_file = json_files[idx] if 0 <= idx < len(json_files) else json_files[0]

    # Carica documento
    with open(test_file, 'r', encoding='utf-8') as f:
        doc = json.load(f)

    title = doc.get('web_metadata', {}).get('title', 'N/A')
    abstract = doc.get('web_metadata', {}).get('abstract', '')

    print(f"\n{'='*80}")
    print(f"DOCUMENTO TEST")
    print(f"{'='*80}")
    print(f"Titolo: {title}")
    print(f"Abstract: {abstract[:200]}...")
    print(f"{'='*80}\n")

    # ESEGUI ENRICHMENT
    enriched = enrich_document_enhanced(
        doc,
        llm_client,
        law_extractor,
        keyword_extractor,
        keyword_refiner,
        confidence_calculator,
        logger
    )

    if enriched:
        meta = enriched['semantic_metadata']
        conf = meta['confidence']

        print(f"\n{'='*80}")
        print("RISULTATI ENRICHMENT")
        print(f"{'='*80}\n")

        print(f"SINTESI:")
        print(f"   {meta['sintesi']}\n")

        print(f"CATEGORIA: {meta['categoria_principale']}")
        if meta.get('categorie_secondarie'):
            print(f"   Secondarie: {', '.join(meta['categorie_secondarie'])}\n")
        else:
            print()

        print(f" LEGGI ({len(meta['articoli_legge'])}):")
        for law in meta['articoli_legge']:
            print(f"   • {law}")
        print(f"   Fonte: {meta['extraction_sources']['laws']}\n")

        print(f"KEYWORDS ({len(meta['parole_chiave'])}):")
        print(f"   {', '.join(meta['parole_chiave'])}\n")

        print(f"PROFESSIONI ({len(meta.get('categorie_professionali', []))}):")
        for prof in meta.get('categorie_professionali', []):
            print(f"   • {prof}")
        print()

        print(f"TEMI:")
        for tema in meta['temi_principali']:
            print(f"   • {tema}")

        print(f"\n{'='*80}")
        print("CONFIDENCE ANALYSIS")
        print(f"{'='*80}")
        print(f"Overall Score: {conf['overall_score']:.3f}")
        print(f"Level: {conf['confidence_level']}")
        print(f"Needs Review: {'YES' if conf['needs_review'] else 'NO'}\n")

        print("Component Scores:")
        for component, score in conf['component_scores'].items():
            bar = '█' * int(score * 20)
            print(f"  {component:25s}: {bar:20s} {score:.3f}")

        if conf['review_reasons']:
            print(f"\nReview Reasons:")
            for reason in conf['review_reasons']:
                print(f"  • {reason}")

        print(f"\n{'='*80}\n")

        # Salva test
        test_path = EnrichmentConfig.ENRICHED_DIR / f"TEST_{test_file.name}"
        with open(test_path, 'w', encoding='utf-8') as f:
            json.dump(enriched, f, ensure_ascii=False, indent=2)

        print(f"Test salvato: {test_path.name}")
        print(f"Log: {log_file}\n")

    else:
        print("ENRICHMENT FALLITO\n")

In [16]:
# MAIN

def main():
    """Funzione principale per batch completo"""

    print("\n" + "="*80)
    print("INAIL - SISTEMA DI ENRICHMENT IBRIDO")
    print("="*80)
    print("\n CARATTERISTICHE:")
    print("   • TF-IDF: Veloce, affidabile, keyword tecniche")
    print("   • LLM Refiner: Bilancia tecnico + generale")
    print("   • Regex Laws: Estrazione normative precisa")
    print("   • Confidence: Score oggettivo multi-componente\n")
    print("="*80 + "\n")

    EnrichmentConfig.setup()
    logger, log_file = setup_logging(EnrichmentConfig.LOG_DIR)

    # Setup Ollama
    logger.info("Setup Ollama...")
    if not install_ollama() or not start_ollama():
        logger.error("Ollama setup fallito")
        return

    model = pull_best_model()
    if not model:
        logger.error("Nessun modello disponibile")
        return

    llm_client = OllamaClient(model=model)

    # Setup componenti
    logger.info("Inizializzazione componenti...")
    law_extractor = ItalianLawExtractor()
    keyword_extractor = TFIDFKeywordExtractor()
    keyword_refiner = LLMKeywordRefiner(model=model)
    confidence_calculator = ConfidenceCalculator()
    logger.info("Setup completato\n")

    # Configurazione batch
    max_docs = input("Max documenti da processare (Enter = tutti): ").strip()
    max_docs = int(max_docs) if max_docs else None

    print(f"\n{'='*80}")
    print("RIEPILOGO:")
    print(f"{'='*80}")
    print(f"LLM: {llm_client.provider}/{llm_client.model}")
    print(f"Keyword: TF-IDF + LLM Refiner (1 chiamata/doc)")
    print(f"Max documenti: {max_docs if max_docs else '(tutti)'}")
    print(f"Output: {EnrichmentConfig.ENRICHED_DIR}")
    print(f"Log: {log_file}")
    print(f"{'='*80}\n")

    confirm = input("Procedere con il batch? (y/n): ").strip().lower()
    if confirm != 'y':
        print("Operazione annullata")
        return

    # ESEGUI BATCH
    stats = batch_enrich_documents(
        llm_client,
        law_extractor,
        keyword_extractor,
        keyword_refiner,
        confidence_calculator,
        max_docs,
        logger
    )

    print(f"\n COMPLETATO!")
    print(f"Log completo: {log_file}")
    print(f"File arricchiti: {EnrichmentConfig.ENRICHED_DIR}\n")

# **Test documento lungo e molto dettagliato**

In [17]:
# TEST 1

if __name__ == "__main__":
    print("\n" + "="*80)
    print("MODALITÀ:")
    print("="*80)
    print("1. Test rapido (1 documento)")
    print("2. Batch completo")
    print("="*80 + "\n")

    mode = input("Scelta (1-2, default 1): ").strip() or "1"

    if mode == "2":
        main()
    else:
        quick_test()


MODALITÀ:
1. Test rapido (1 documento)
2. Batch completo

Scelta (1-2, default 1): 1

TEST SISTEMA ENRICHMENT - TF-IDF + LLM HYBRID

Configurazione LLM...
[Setup] Installazione Ollama...
Ollama installato
[Setup] Avvio server Ollama...
Server avviato
[Setup] Download modello LLM...
  Tentando mistral:latest...
  mistral:latest pronto!

[Setup] Inizializzazione componenti...
[TF-IDF] Inizializzazione...
[TF-IDF] ✓ Pronto
Tutti i componenti pronti

Documenti disponibili: 385

1. 11__Rapporto_sull_attività_di_accertamento_tecnico_20251030_200042.js
2. Abbandoni_superficiali_di_manufatti_in_cemento_ami_20251105_001849.jso
3. Aderenza_agli_standard_di_sicurezza_in_risonanza_m_20251106_233622.jso
4. Agenti_biologici__fattori_di_rischio_cancerogeno_o_20251108_205111.jso
5. Agenti_cancerogeni_e_mutageni._Lavorare_sicuri_-_e_20251105_014750.jso
6. Agenti_chimici_pericolosi_-_Istruzioni_ad_uso_dei__20251107_015421.jso
7. Aggiornamento_della_stima_dei_lavoratori_potenzial_20251104_233455.jso
8.

# **Test documento senza leggi citate**

In [19]:
# TEST 3

if __name__ == "__main__":
    print("\n" + "="*80)
    print("MODALITÀ:")
    print("="*80)
    print("1. Test rapido (1 documento)")
    print("2. Batch completo")
    print("="*80 + "\n")

    mode = input("Scelta (1-2, default 1): ").strip() or "1"

    if mode == "2":
        main()
    else:
        quick_test()


MODALITÀ:
1. Test rapido (1 documento)
2. Batch completo

Scelta (1-2, default 1): 1

TEST SISTEMA ENRICHMENT - TF-IDF + LLM HYBRID

Configurazione LLM...
[Setup] Installazione Ollama...
Ollama installato
[Setup] Avvio server Ollama...
Server avviato
[Setup] Download modello LLM...
  Tentando mistral:latest...
  mistral:latest pronto!

[Setup] Inizializzazione componenti...
[TF-IDF] Inizializzazione...
[TF-IDF] ✓ Pronto
Tutti i componenti pronti

Documenti disponibili: 385

1. 11__Rapporto_sull_attività_di_accertamento_tecnico_20251030_200042.js
2. Abbandoni_superficiali_di_manufatti_in_cemento_ami_20251105_001849.jso
3. Aderenza_agli_standard_di_sicurezza_in_risonanza_m_20251106_233622.jso
4. Agenti_biologici__fattori_di_rischio_cancerogeno_o_20251108_205111.jso
5. Agenti_cancerogeni_e_mutageni._Lavorare_sicuri_-_e_20251105_014750.jso
6. Agenti_chimici_pericolosi_-_Istruzioni_ad_uso_dei__20251107_015421.jso
7. Aggiornamento_della_stima_dei_lavoratori_potenzial_20251104_233455.jso
8.

In [18]:
# TEST 2

if __name__ == "__main__":
    print("\n" + "="*80)
    print("MODALITÀ:")
    print("="*80)
    print("1. Test rapido (1 documento)")
    print("2. Batch completo")
    print("="*80 + "\n")

    mode = input("Scelta (1-2, default 1): ").strip() or "1"

    if mode == "2":
        main()
    else:
        quick_test()


MODALITÀ:
1. Test rapido (1 documento)
2. Batch completo

Scelta (1-2, default 1): 1

TEST SISTEMA ENRICHMENT - TF-IDF + LLM HYBRID

Configurazione LLM...
[Setup] Installazione Ollama...
Ollama installato
[Setup] Avvio server Ollama...
Server avviato
[Setup] Download modello LLM...
  Tentando mistral:latest...
  mistral:latest pronto!

[Setup] Inizializzazione componenti...
[TF-IDF] Inizializzazione...
[TF-IDF] ✓ Pronto
Tutti i componenti pronti

Documenti disponibili: 385

1. 11__Rapporto_sull_attività_di_accertamento_tecnico_20251030_200042.js
2. Abbandoni_superficiali_di_manufatti_in_cemento_ami_20251105_001849.jso
3. Aderenza_agli_standard_di_sicurezza_in_risonanza_m_20251106_233622.jso
4. Agenti_biologici__fattori_di_rischio_cancerogeno_o_20251108_205111.jso
5. Agenti_cancerogeni_e_mutageni._Lavorare_sicuri_-_e_20251105_014750.jso
6. Agenti_chimici_pericolosi_-_Istruzioni_ad_uso_dei__20251107_015421.jso
7. Aggiornamento_della_stima_dei_lavoratori_potenzial_20251104_233455.jso
8.